[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/autoregressive_generator.ipynb)

# Autoregressive Models for Image Generation

An autoregressive model generates an image **one pixel at a time**, in raster order (row by row, left to right). It is based on the chain rule of probability:

$$p(x) = p(x_1)\, p(x_2 \mid x_1) \cdots p(x_n \mid x_1, \dots, x_{n-1}) = \prod_{i=1}^{n} p(x_i \mid x_{<i})$$

A neural network receives the pixels generated so far, $x_{<i}$, and outputs a probability for every possible colour of the next pixel, $x_i$. In this notebook we build a **Gated PixelCNN** (van den Oord et al., 2016) and train it on the same $16 \times 16$ sprites used in the Diffusion Models lecture.

**Exercises** in this notebook (look for `# TODO`):

1. Complete the masks of the masked convolutions;
2. Complete the loss function used in training;
3. Complete the sampling loop;
4. Answer the questions at the end.

In [ ]:
!pip install -q kagglehub

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print('Using {} device'.format(device))

# Dataset

The sprites are $16 \times 16$ RGB images, each with a context vector of 5 categories (one-hot). The dataset repeats the same sprites many times, so we keep only one copy of each.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("ebrahimelgazar/pixel-art")
sprites = np.load(path + "/sprites.npy")
labels = np.load(path + "/sprites_labels.npy")
print("sprites:", sprites.shape, "labels:", labels.shape)

# keep one copy of each sprite
_, idx = np.unique(sprites.reshape(len(sprites), -1), axis=0, return_index=True)
idx = np.sort(idx)
sprites, labels = sprites[idx], labels[idx]
print("unique sprites:", sprites.shape)

CATEGORIES = ['human', 'non-human', 'food', 'spell', 'side-facing']
HEIGHT = 16  # 16x16 images

In [ ]:
rng = np.random.default_rng(0)
_, axs = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, i in zip(axs.flatten(), rng.choice(len(sprites), 16, replace=False)):
    ax.imshow(sprites[i])
    ax.set_title(CATEGORIES[labels[i].argmax()], fontsize=8)
    ax.axis('off')
plt.show()

# A palette of colours

Each pixel will be a **category**: the network ends with a softmax over the possible colours. With $256^3$ RGB colours this would be impossible, so, as in Image GPT, the colours are reduced to a palette of `N_COLORS` colours with k-means. Each pixel becomes the index of the closest colour of the palette.

In [ ]:
N_COLORS = 64  # size of the palette


def fit_palette(sprites, n_colors=N_COLORS, n_iter=30, n_pixels=200_000, seed=0):
    # K-means on the RGB values of the sprites (N x H x W x 3, uint8)
    g = torch.Generator().manual_seed(seed)
    pixels = torch.as_tensor(sprites).reshape(-1, 3).float()
    pixels = pixels[torch.randint(len(pixels), (n_pixels,), generator=g)]
    palette = pixels[torch.randperm(len(pixels), generator=g)[:n_colors]]
    for _ in range(n_iter):
        idx = torch.cdist(pixels, palette).argmin(dim=1)
        for k in range(n_colors):
            members = pixels[idx == k]
            if len(members) > 0:
                palette[k] = members.mean(dim=0)
    return palette


def to_indices(images, palette):
    # RGB images (B x H x W x 3, uint8) -> palette indices (B x H x W)
    pixels = torch.as_tensor(images).reshape(-1, 3).float()
    idx = torch.cdist(pixels, palette.cpu()).argmin(dim=1)
    return idx.reshape(images.shape[:-1])


def to_rgb(indices, palette):
    # palette indices (B x H x W) -> RGB images (B x H x W x 3, values in [0, 1])
    return (palette.to(indices.device)[indices] / 255).cpu()


palette = fit_palette(sprites)

plt.figure(figsize=(10, 1))
plt.imshow(palette[None] / 255)
plt.axis('off')
plt.title('Palette')
plt.show()

In [ ]:
# original sprites (top) and sprites with the palette colours (bottom)
X = to_indices(sprites, palette)
_, axs = plt.subplots(2, 8, figsize=(12, 3))
for k, i in enumerate(range(0, len(sprites), len(sprites) // 8)[:8]):
    axs[0, k].imshow(sprites[i])
    axs[1, k].imshow(to_rgb(X[i], palette))
for ax in axs.flatten():
    ax.axis('off')
plt.show()

In [ ]:
# 90% of the sprites for training, 10% for testing
C = torch.tensor(labels).float()
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
n_train = int(0.9 * len(X))
X_train, C_train = X[perm[:n_train]].to(device), C[perm[:n_train]].to(device)
X_test, C_test = X[perm[n_train:]].to(device), C[perm[n_train:]].to(device)
print("train:", X_train.shape, "test:", X_test.shape)

# Masked convolutions

To train on all the pixels at once, the prediction for pixel $x_i$ must never depend on $x_i$ or on the pixels after it. The weights of the kernels are multiplied by a **mask** with zeros on the "future" positions.

A single masked convolution has a **blind spot** (some pixels above and to the right are never seen). The Gated PixelCNN avoids it with two stacks:

* the **vertical stack** sees all the rows **above** the current pixel: a $k \times k$ kernel whose rows below the centre are zero;
* the **horizontal stack** sees the pixels **to the left**, in the current row: a $1 \times k$ kernel whose positions to the right of the centre are zero.

In the first layer, the centre must also be hidden (`mask_center=True`, mask A): the vertical stack ignores the whole centre row, and the horizontal stack ignores the centre pixel. In the other layers the centre already holds information about previous pixels only, so it can be used (mask B).

## Exercise 1

Complete the masks of `VerticalStackConv` and `HorizontalStackConv`. For `kernel_size=3`, the expected masks are:

| | `mask_center=True` | `mask_center=False` |
|---|---|---|
| vertical | `[[1,1,1],[0,0,0],[0,0,0]]` | `[[1,1,1],[1,1,1],[0,0,0]]` |
| horizontal | `[[1,0,0]]` | `[[1,1,0]]` |

In [ ]:
class MaskedConv2d(nn.Conv2d):
    # convolution whose kernel is multiplied by a fixed mask

    def __init__(self, in_channels, out_channels, mask, dilation=1):
        kernel_size = mask.shape
        padding = tuple(dilation * (k - 1) // 2 for k in kernel_size)
        super().__init__(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.register_buffer('mask', mask[None, None])

    def forward(self, x):
        return F.conv2d(x, self.weight * self.mask, self.bias, self.stride, self.padding, self.dilation)


class VerticalStackConv(MaskedConv2d):
    # sees all the rows above the current pixel (and the current row, if mask_center is False)

    def __init__(self, in_channels, out_channels, kernel_size=3, mask_center=False, dilation=1):
        mask = torch.ones(kernel_size, kernel_size)
        # TODO: set to 0 the rows below the centre and, if mask_center, the centre row
        super().__init__(in_channels, out_channels, mask, dilation)


class HorizontalStackConv(MaskedConv2d):
    # sees the pixels to the left of the current one (and the current one, if mask_center is False)

    def __init__(self, in_channels, out_channels, kernel_size=3, mask_center=False, dilation=1):
        mask = torch.ones(1, kernel_size)
        # TODO: set to 0 the positions right of the centre and, if mask_center, the centre
        super().__init__(in_channels, out_channels, mask, dilation)

In [ ]:
for mask_center in [True, False]:
    print(f"mask_center={mask_center}")
    print("vertical:\n", VerticalStackConv(1, 1, mask_center=mask_center).mask[0, 0])
    print("horizontal:\n", HorizontalStackConv(1, 1, mask_center=mask_center).mask[0, 0])

# Gated PixelCNN

Each layer combines the two stacks with a **gated activation**, as in LSTMs, and adds the context $h$ (the category of the sprite):

$$y = \tanh(W_f * x + V_f^\top h) \odot \sigma(W_g * x + V_g^\top h)$$

The vertical stack only sees rows above, so its features can be passed to the horizontal stack. The horizontal stack has a residual connection. The last layer outputs `N_COLORS` logits for each pixel.

In [ ]:
class GatedMaskedConv(nn.Module):

    def __init__(self, channels, n_cfeat, dilation=1):
        super().__init__()
        self.conv_vert = VerticalStackConv(channels, 2 * channels, dilation=dilation)
        self.conv_horiz = HorizontalStackConv(channels, 2 * channels, dilation=dilation)
        self.conv_vert_to_horiz = nn.Conv2d(2 * channels, 2 * channels, kernel_size=1)
        self.conv_horiz_1x1 = nn.Conv2d(channels, channels, kernel_size=1)
        self.cond_vert = nn.Linear(n_cfeat, 2 * channels)
        self.cond_horiz = nn.Linear(n_cfeat, 2 * channels)

    def forward(self, v_stack, h_stack, c):
        v_feat = self.conv_vert(v_stack)
        v_val, v_gate = (v_feat + self.cond_vert(c)[:, :, None, None]).chunk(2, dim=1)
        v_out = torch.tanh(v_val) * torch.sigmoid(v_gate)

        h_feat = self.conv_horiz(h_stack) + self.conv_vert_to_horiz(v_feat)
        h_val, h_gate = (h_feat + self.cond_horiz(c)[:, :, None, None]).chunk(2, dim=1)
        h_out = self.conv_horiz_1x1(torch.tanh(h_val) * torch.sigmoid(h_gate))
        return v_out, h_out + h_stack  # residual connection


class PixelCNN(nn.Module):
    # input: palette indices (B x H x W); output: logits (B x n_colors x H x W)

    def __init__(self, n_colors=N_COLORS, channels=64, n_cfeat=5, palette=None):
        super().__init__()
        self.n_cfeat = n_cfeat
        self.register_buffer('palette', palette if palette is not None else torch.zeros(n_colors, 3))
        self.embedding = nn.Embedding(n_colors, channels)
        # first layer: the centre is hidden (mask A)
        self.conv_vstack = VerticalStackConv(channels, channels, mask_center=True)
        self.conv_hstack = HorizontalStackConv(channels, channels, mask_center=True)
        # other layers (mask B); the dilation grows the receptive field
        self.layers = nn.ModuleList([GatedMaskedConv(channels, n_cfeat, dilation=d) for d in (1, 2, 1, 2, 1, 2, 1)])
        self.conv_out = nn.Conv2d(channels, n_colors, kernel_size=1)

    def forward(self, x, c=None):
        if c is None:  # no context
            c = torch.zeros(x.shape[0], self.n_cfeat, device=x.device)
        x = self.embedding(x).permute(0, 3, 1, 2)
        v_stack = self.conv_vstack(x)
        h_stack = self.conv_hstack(x)
        for layer in self.layers:
            v_stack, h_stack = layer(v_stack, h_stack, c)
        return self.conv_out(F.elu(h_stack))

In [ ]:
torch.manual_seed(0)
model = PixelCNN(palette=palette).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

Let's check the masks: if we change pixel $(8, 8)$, only the predictions of the **following** pixels may change (white). The pixels before it, including $(8, 0) \dots (8, 8)$, must stay black.

In [ ]:
x = torch.randint(0, N_COLORS, (1, HEIGHT, HEIGHT), device=device)
x2 = x.clone()
x2[0, 8, 8] = (x[0, 8, 8] + 1) % N_COLORS

with torch.no_grad():
    diff = (model(x) - model(x2)).abs().sum(dim=1)[0]

plt.imshow(diff.cpu() > 1e-3, cmap='gray')
plt.title('Predictions changed by pixel (8, 8)')
plt.show()

# Training

The model is trained by maximum likelihood: the loss is the negative log-likelihood of the real pixels,

$$\mathcal{L}(\theta) = -\sum_{i=1}^{n} \log p_\theta(x_i \mid x_{<i})$$

which is a **cross-entropy** between the real colour of each pixel and the predicted distribution. Thanks to the masks, all the pixels are predicted at once, in a single forward pass (teacher forcing). The loss is shown in **bits per pixel**.

As in the diffusion model, the context is removed in 10% of the examples, so that the model also learns to generate without context.

## Exercise 2

Complete the training loop: compute the `logits` of the model for the batch `x` with context `c`, and the `loss` (use `F.cross_entropy`; note that it accepts logits with shape `(B, K, H, W)` and targets with shape `(B, H, W)`).

In [ ]:
MODEL_FILE = Path("autoregressive_generator.pth")
N_EPOCHS = 40
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

train_bpp, test_bpp = [], []

if MODEL_FILE.is_file():
    model.load_state_dict(torch.load(MODEL_FILE, map_location=device, weights_only=True))
    print("Model loaded from", MODEL_FILE)
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    for epoch in range(N_EPOCHS):
        model.train()
        optimizer.param_groups[0]["lr"] = LEARNING_RATE * (1 - epoch / N_EPOCHS)  # linear decay
        total = 0
        for b in torch.randperm(n_train, device=device).split(BATCH_SIZE):
            x = X_train[b]
            # randomly remove the context
            c = C_train[b] * torch.bernoulli(torch.full((len(b), 1), 0.9, device=device))

            logits = ...  # TODO
            loss = ...  # TODO

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(b)

        model.eval()
        with torch.no_grad():
            test_loss = F.cross_entropy(model(X_test, C_test), X_test).item()
        train_bpp.append(total / n_train / np.log(2))
        test_bpp.append(test_loss / np.log(2))
        print(f"epoch {epoch}: train {train_bpp[-1]:.3f}, test {test_bpp[-1]:.3f} bits per pixel")

    torch.save(model.state_dict(), MODEL_FILE)

model.eval();

In [ ]:
if train_bpp:
    plt.plot(train_bpp, label='train')
    plt.plot(test_bpp, label='test')
    plt.xlabel('epoch')
    plt.ylabel('bits per pixel')
    plt.legend()
    plt.show()

# Sampling

Generation starts with an empty canvas (pixels with value `-1`). For each pixel, in raster order:

1. the network receives the image generated so far and predicts the logits of every pixel;
2. the logits of the current pixel are divided by the **temperature** and converted into probabilities with a softmax;
3. a colour is sampled from these probabilities (`torch.multinomial`) and written in the image.

The pixels that are already given (`start`) are kept, which allows the model to complete images.

## Exercise 3

Complete the sampling loop.

In [ ]:
@torch.no_grad()
def sample(model, n_sample, c=None, temperature=1.0, start=None, save_rate=None):
    # start: (B x H x W) palette indices; the pixels with -1 are generated, the others are kept
    if start is None:
        samples = torch.full((n_sample, HEIGHT, HEIGHT), -1, dtype=torch.long, device=device)
    else:
        samples = start.clone().to(device)

    intermediate = []  # images during the generation
    for r in range(HEIGHT):
        for col in range(HEIGHT):
            if save_rate and (r * HEIGHT + col) % save_rate == 0:
                intermediate.append(samples.clone())
            todo = samples[:, r, col] == -1  # images where this pixel must be generated
            if not todo.any():
                continue
            # TODO: compute the logits (hint: replace -1 by a valid colour with samples.clamp(min=0)),
            #       the probabilities of pixel (r, col) with temperature, and sample one colour per image
            new = ...
            samples[todo, r, col] = new[todo]
    intermediate.append(samples.clone())
    return samples, intermediate

In [ ]:
def show_images(samples, nrow=1, titles=None):
    imgs = to_rgb(samples.clamp(min=0), model.palette)
    imgs[samples.cpu() == -1] = 0.85  # pixels not generated yet in grey
    ncol = len(imgs) // nrow
    _, axs = plt.subplots(nrow, ncol, figsize=(1.2 * ncol, 1.2 * nrow + 0.3 * (titles is not None)))
    for i, ax in enumerate(np.array(axs).flatten()):
        ax.imshow(imgs[i])
        ax.axis('off')
        if titles is not None:
            ax.set_title(titles[i], fontsize=8)
    plt.show()

In [ ]:
# two sprites of each category
ctx = torch.eye(5, device=device).repeat_interleave(2, dim=0)
samples, _ = sample(model, len(ctx), ctx)
show_images(samples, titles=[CATEGORIES[k] for k in ctx.argmax(dim=1)])

In [ ]:
# without context
samples, _ = sample(model, 10)
show_images(samples)

## Watching the generation

Unlike diffusion, where all the pixels are refined together, each pixel is decided once and never changed.

In [ ]:
ctx = torch.eye(5, device=device)[[0, 1, 2]]
samples, intermediate = sample(model, len(ctx), ctx, save_rate=32)
for steps in torch.stack(intermediate, dim=1):
    show_images(steps)

## Temperature

With $\tau < 1$ the distributions are sharper; with $\tau > 1$ they are flatter.

In [ ]:
ctx = torch.eye(5, device=device)[[0] * 10]  # 'human'
for temperature in [0.5, 0.8, 1.0, 1.5]:
    torch.manual_seed(0)
    samples, _ = sample(model, len(ctx), ctx, temperature=temperature)
    print(f"temperature = {temperature}")
    show_images(samples)

## Completing an image

Sprites from the **test set** (never seen during training): the top half is kept and the bottom half is generated 4 times.

In [ ]:
x, c = X_test[:4], C_test[:4]
start = x.repeat_interleave(4, dim=0)
start[:, HEIGHT // 2:] = -1  # pixels to generate
completions, _ = sample(model, len(start), c.repeat_interleave(4, dim=0), start=start)

half = x.clone()
half[:, HEIGHT // 2:] = -1
# each row: original, top half, 4 completions
rows = torch.cat([x[:, None], half[:, None], completions.view(4, 4, HEIGHT, HEIGHT)], dim=1)
show_images(rows.flatten(0, 1), nrow=4)

# Exact likelihood

An autoregressive model computes $p(x)$ exactly, so we can measure how likely an image is, in bits per pixel.

In [ ]:
@torch.no_grad()
def bits_per_pixel(model, x, c=None):
    nll = F.cross_entropy(model(x, c), x, reduction='none').sum(dim=(1, 2))  # -log p(x) of each image
    return nll / (HEIGHT * HEIGHT) / np.log(2)


shuffled = X_test.flatten(1)[:, torch.randperm(HEIGHT * HEIGHT)].view_as(X_test)  # same colours, wrong place
noise = torch.randint(0, N_COLORS, X_test.shape, device=device)
for name, imgs in [('train sprites', X_train), ('test sprites', X_test), ('shuffled pixels', shuffled), ('random noise', noise)]:
    print(f"{name:16s} {bits_per_pixel(model, imgs).mean():.2f} bits per pixel")

# Questions

1. How many forward passes of the network are needed to generate one sprite? And to generate a $256 \times 256$ RGB image with 256 values per channel? Why is training much faster than sampling?
2. Why must the centre of the kernel be hidden in the first layer, but not in the following layers?
3. What is the effect of the temperature on the generated sprites? What happens when $\tau \to 0$?
4. Why do the shuffled pixels have a much higher number of bits per pixel than the real sprites, although they have exactly the same colours? Why is the value for the test sprites higher than for the training sprites?
5. Compare this model with the diffusion model: number of steps, what each step does, loss function, and the possibility of completing an image.